# aFRR VWAP Validation (Regelleistung 15-min)

This notebook validates hourly aFRR VWAP against 15-minute price/volume rows for two windows:

1. **Pre-PICASSO:** 2021-03-01 10:00 to 16:00 UTC  
2. **Post-PICASSO:** 2022-08-01 10:00 to 16:00 UTC

Hourly VWAP is repeated on each quarter-hour row (same hour) for visual comparison.

In [ ]:
from pathlib import Path
import polars as pl
import pandas as pd

PATH_15M = Path("data/raw/regelleistung_15min/afrr_price_volume_15min.parquet")
if not PATH_15M.exists():
    raise FileNotFoundError(f"Missing 15-min parquet: {PATH_15M}")

df = pl.read_parquet(PATH_15M).with_columns(
    pl.col("timestamp_utc").cast(pl.Datetime(time_unit="us", time_zone="UTC"), strict=False)
).sort("timestamp_utc")

print("Rows:", df.height)
print("Columns:", df.columns)

In [ ]:
def first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


price_pos_col = first_existing(
    df.columns,
    [
        "afrr_avg_activation_price_pos",
        "afrr_activation_avg_price_pos",
        "afrr_marginal_activation_price_pos",
        "afrr_activation_marginal_price_pos",
    ],
)
price_neg_col = first_existing(
    df.columns,
    [
        "afrr_avg_activation_price_neg",
        "afrr_activation_avg_price_neg",
        "afrr_marginal_activation_price_neg",
        "afrr_activation_marginal_price_neg",
    ],
)
vol_pos_col = first_existing(df.columns, ["afrr_activated_mw_pos", "activated_volume_pos_mw"])
vol_neg_col = first_existing(df.columns, ["afrr_activated_mw_neg", "activated_volume_neg_mw"])

print("price_pos_col:", price_pos_col)
print("price_neg_col:", price_neg_col)
print("vol_pos_col:", vol_pos_col)
print("vol_neg_col:", vol_neg_col)


def build_window_table(df_in: pl.DataFrame, start_utc: str, end_utc: str) -> pd.DataFrame:
    start_ts = pd.Timestamp(start_utc)
    end_ts = pd.Timestamp(end_utc)

    w = df_in.filter(
        (pl.col("timestamp_utc") >= pl.lit(start_ts))
        & (pl.col("timestamp_utc") < pl.lit(end_ts))
    ).sort("timestamp_utc")

    needed = ["timestamp_utc"]
    for c in [vol_pos_col, vol_neg_col, price_pos_col, price_neg_col]:
        if c is not None:
            needed.append(c)
    w = w.select(needed)

    exprs = [pl.col("timestamp_utc").dt.truncate("1h").alias("hour_utc")]
    if price_pos_col and vol_pos_col:
        exprs.append(
            (
                pl.col(price_pos_col).cast(pl.Float64, strict=False)
                * pl.col(vol_pos_col).cast(pl.Float64, strict=False)
            ).alias("__wc_pos")
        )
    if price_neg_col and vol_neg_col:
        exprs.append(
            (
                pl.col(price_neg_col).cast(pl.Float64, strict=False)
                * pl.col(vol_neg_col).cast(pl.Float64, strict=False)
            ).alias("__wc_neg")
        )

    w2 = w.with_columns(exprs)

    agg_exprs = []
    if price_pos_col and vol_pos_col:
        agg_exprs += [
            pl.sum("__wc_pos").alias("sum_wc_pos"),
            pl.sum(pl.col(vol_pos_col).cast(pl.Float64, strict=False)).alias("sum_vol_pos"),
            pl.mean(pl.col(price_pos_col).cast(pl.Float64, strict=False)).alias("mean_price_pos"),
        ]
    if price_neg_col and vol_neg_col:
        agg_exprs += [
            pl.sum("__wc_neg").alias("sum_wc_neg"),
            pl.sum(pl.col(vol_neg_col).cast(pl.Float64, strict=False)).alias("sum_vol_neg"),
            pl.mean(pl.col(price_neg_col).cast(pl.Float64, strict=False)).alias("mean_price_neg"),
        ]

    hourly = w2.group_by("hour_utc").agg(agg_exprs).sort("hour_utc")

    if price_pos_col and vol_pos_col:
        hourly = hourly.with_columns(
            pl.when(pl.col("sum_vol_pos") != 0)
            .then(pl.col("sum_wc_pos") / pl.col("sum_vol_pos"))
            .otherwise(pl.col("mean_price_pos"))
            .alias("afrr_vwap_pos_eur_mwh")
        )
    if price_neg_col and vol_neg_col:
        hourly = hourly.with_columns(
            pl.when(pl.col("sum_vol_neg") != 0)
            .then(pl.col("sum_wc_neg") / pl.col("sum_vol_neg"))
            .otherwise(pl.col("mean_price_neg"))
            .alias("afrr_vwap_neg_eur_mwh")
        )

    joined = w2.join(
        hourly.select(
            [
                c
                for c in ["hour_utc", "afrr_vwap_pos_eur_mwh", "afrr_vwap_neg_eur_mwh"]
                if c in hourly.columns
            ]
        ),
        on="hour_utc",
        how="left",
    ).sort("timestamp_utc")

    rename = {}
    if vol_pos_col:
        rename[vol_pos_col] = "vol_15m_pos_mw"
    if vol_neg_col:
        rename[vol_neg_col] = "vol_15m_neg_mw"
    if price_pos_col:
        rename[price_pos_col] = "price_15m_pos_eur_mwh"
    if price_neg_col:
        rename[price_neg_col] = "price_15m_neg_eur_mwh"

    out = joined.rename(rename)
    out_cols = [
        "timestamp_utc",
        "vol_15m_pos_mw",
        "vol_15m_neg_mw",
        "price_15m_pos_eur_mwh",
        "price_15m_neg_eur_mwh",
        "afrr_vwap_pos_eur_mwh",
        "afrr_vwap_neg_eur_mwh",
    ]
    out_cols = [c for c in out_cols if c in out.columns]
    return out.select(out_cols).to_pandas().set_index("timestamp_utc")


In [ ]:
pre_table = build_window_table(df, "2021-03-01T10:00:00Z", "2021-03-01T16:00:00Z")
post_table = build_window_table(df, "2022-08-01T10:00:00Z", "2022-08-01T16:00:00Z")

print("Pre-PICASSO window (2021-03-01 10:00-16:00 UTC)")
display(pre_table)

print("Post-PICASSO window (2022-08-01 10:00-16:00 UTC)")
display(post_table)